## Structured output
Models can be requested to provide their response in a format matching a given schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing. LangChain supports multiple schema types and methods for enforcing structured output.

## Pydantic
Pydantic models provide the richest feature set with field validation, descriptions, and nested structures.

In [ ]:
import os 
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

model=init_chat_model("groq:llama-3.3-70b-versatile")
response=model.invoke("Why do parrots talk")
response

AIMessage(content="Parrots are known for their remarkable ability to mimic human speech and other sounds, and scientists have several theories about why they do this. Here are some possible reasons:\n\n1. **Communication and social interaction**: In the wild, parrots use vocalizations to communicate with each other, just like humans do. They may mimic sounds to convey information, warn other birds of predators, or attract a mate. By talking, parrots may be attempting to interact with their human caregivers in a similar way.\n2. **Learning and cognition**: Parrots are highly intelligent birds, and talking may be a way for them to learn and exercise their cognitive abilities. By mimicking sounds, they may be practicing their vocal skills, learning new words, and understanding the relationships between sounds and meanings.\n3. **Attention and affection**: Parrots are social birds that thrive on attention and interaction. By talking, they may be seeking attention and affection from their h

In [3]:
from pydantic import BaseModel,Field

class Movie(BaseModel):
    title:str=Field(description="This is the title of the movie")
    year:int=Field(desciption="this is the year movie was released")
    director:str=Field(description="director of the film")
    rating:float=Field(description="the movies rating out of 10")

C:\Users\Rajat Singh\AppData\Local\Temp\ipykernel_23816\896658611.py:5: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'desciption'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  year:int=Field(desciption="this is the year movie was released")


In [5]:
model_with_structure=model.with_structured_output(Movie)

In [6]:
model_with_structure

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.4.9', 'langchain': '1.3.14'}}, output_version=None, profile={'name': 'Llama 3.3 70B Versatile', 'release_date': '2024-12-06', 'last_updated': '2024-12-06', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 32768, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000001DF05F73B60>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001DF061AC6E0>, model_name='llama-3.3-70b-versatile', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': 

In [7]:
model_with_structure.invoke("Provide details about inception")

Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.5)

## Message output along with parsed structure

In [8]:
from pydantic import BaseModel,Field

class Movie(BaseModel):
    title:str=Field(...,description="This is the title of the movie")
    year:int=Field(...,desciption="this is the year movie was released")
    director:str=Field(...,description="director of the film")
    rating:float=Field(...,description="the movies rating out of 10")

model_with_structure=model.with_structured_output(Movie,include_raw=True)

response=model_with_structure.invoke("Tell me about the movie inception")

C:\Users\Rajat Singh\AppData\Local\Temp\ipykernel_23816\3173869429.py:5: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'desciption'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  year:int=Field(...,desciption="this is the year movie was released")


In [9]:
response

{'raw': AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'nvx8ne5g4', 'function': {'arguments': '{"director":"Christopher Nolan","rating":8.5,"title":"Inception","year":2010}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 32, 'prompt_tokens': 282, 'total_tokens': 314, 'completion_time': 0.042065628, 'completion_tokens_details': None, 'prompt_time': 0.016748871, 'prompt_tokens_details': None, 'queue_time': 0.231114599, 'total_time': 0.058814499}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_3272ea2d91', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019f9403-49aa-7a50-a56f-ba05a91f1dc5-0', tool_calls=[{'name': 'Movie', 'args': {'director': 'Christopher Nolan', 'rating': 8.5, 'title': 'Inception', 'year': 2010}, 'id': 'nvx8ne5g4', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 282, 'output_tokens': 32,

## Nested Structure

In [10]:
from pydantic import BaseModel,Field

class Actor(BaseModel):
    name:str
    role:str

class MovieDetail(BaseModel):
    title:str
    year:int
    cast:list[Actor]
    genres:list[str]
    budget:float | None =Field(None,description="Budget in millions USD")


model_with_structure=model.with_structured_output(MovieDetail)

response=model_with_structure.invoke("proide details about the movie inception")

response

MovieDetail(title='Inception', year=2010, cast=[Actor(name='Leonardo DiCaprio', role='Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur'), Actor(name='Ellen Page', role='Ariadne')], genres=['Action', 'Sci-Fi', 'Thriller'], budget=160.0)

## TypeDict

TypedDict provides a simpler alternative using Python’s built-in typing, ideal when you don’t need runtime validation.

In [11]:
from typing import TypedDict,Annotated


class MovieDict(TypedDict):
    """A movie with details."""
    title: Annotated[str, ..., "The title of the movie"]
    year: Annotated[int, ..., "The year the movie was released"]    
    director: Annotated[str, ..., "The director of the movie"]
    rating: Annotated[float, ..., "The movie's rating out of 10"]

model_withtypedict=model.with_structured_output(MovieDict)
response=model_withtypedict.invoke("please provide the details of the movie avengers")
response

{'director': 'Joss Whedon', 'rating': 8.1, 'title': 'Avengers', 'year': 2012}

In [12]:
from pydantic import BaseModel,Field

class Actor(TypedDict):
    name:str
    role:str

class MovieDetail(TypedDict):
    title:str
    year:int
    cast:list[Actor]
    genres:list[str]
    budget:float | None =Field(None,description="Budget in millions USD")


model_with_structure=model.with_structured_output(MovieDetail)

response=model_with_structure.invoke("proide details about the movie inception")

response

{'budget': 160000000,
 'cast': [{'name': 'Leonardo DiCaprio', 'role': 'Cobb'},
  {'name': 'Joseph Gordon-Levitt', 'role': 'Arthur'},
  {'name': 'Ellen Page', 'role': 'Ariadne'}],
 'genres': ['Action', 'Sci-Fi', 'Thriller'],
 'title': 'Inception',
 'year': 2010}

In [14]:
model.profile

{'name': 'Llama 3.3 70B Versatile',
 'release_date': '2024-12-06',
 'last_updated': '2024-12-06',
 'open_weights': True,
 'max_input_tokens': 131072,
 'max_output_tokens': 32768,
 'text_inputs': True,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'text_outputs': True,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': False,
 'tool_calling': True,
 'attachment': False,
 'temperature': True}

## Data Classes

A data class is a class typically containing mainly data, although there aren’t really any restrictions. You create it using the @dataclass decorator

In [ ]:
import os
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

In [16]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent
from langchain_groq import ChatGroq

llm=ChatGroq(
    model="llama-3.3-70b-versatile"
)
class ContactInfo(BaseModel):
    """Contact information for a person."""
    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="The phone number of the person")

agent = create_agent(
    model=llm,
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result

{'messages': [HumanMessage(content='Extract contact info from: John Doe, john@example.com, (555) 123-4567', additional_kwargs={}, response_metadata={}, id='6f97471c-caac-46d5-aed4-0533567cb046'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'rp2kxzt99', 'function': {'arguments': '{"email":"john@example.com","name":"John Doe","phone":"(555) 123-4567"}', 'name': 'ContactInfo'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 33, 'prompt_tokens': 288, 'total_tokens': 321, 'completion_time': 0.063065637, 'completion_tokens_details': None, 'prompt_time': 0.038192252, 'prompt_tokens_details': None, 'queue_time': 0.200234808, 'total_time': 0.101257889}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_3272ea2d91', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019f9410-2d1c-7cc0-ab15-04836a7a8a3e-0', tool_calls=[{'name': 'ContactInfo', 'args': {'email': 'j

In [17]:
print(result['structured_response'])

name='John Doe' email='john@example.com' phone='(555) 123-4567'


In [18]:
from dataclasses import dataclass
from langchain.agents import create_agent


@dataclass
class ContactInfo:
    """Contact information for a person."""
    name: str 
    email: str 
    phone: str 

llm=ChatGroq(
    model="llama-3.3-70b-versatile"
)

agent = create_agent(
    model=llm,
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result["structured_response"]

ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')